# AG-HYPOPT · Trial 01

> ▶ RUN = execute this cell, never modify it  ·  ✍️ WRITE = replace the [placeholder] with your text  ·  ✍️ SET = put your value where the [placeholder] is, then run the cell. Full rules: `instructions.md`


## 1. ✍️ WRITE - Read and summarize

Read `context.md` (physics/model knowledge). If there are previous trials, read them too
(all `trial_XX` notebooks numbered lower than this one) and `trials.json` (the registry).
The current best is the recorded trial with the lowest objective; it is the reference
for the cell-3 choice.

This is the first trial of experiment_2, a vanilla test of the AG-HYPOPT algorithm. There are no previous trials and the registry is empty, so there is no current best and no prior history to compare against. Setup: 8 synthetic experiments (1nW and 3nW at Trans05, Trans20, Trans60, Trans100), targets at the true values with fixed seeds; a search space of 4 tunables (sigma_ref, lr_mu, lr_gamma, gamma_anneal); n_runs = 100 and n_iter = 100 fixed; roughly 30 minutes per trial. This trial is the first of the 5 uniform warm-up trials, so the candidate table it produces is entirely random by design, and the model only starts proposing from trial 6 onward.


In [1]:
# 2. ▶ RUN - Propose candidate trials (AGHyperopt)
TRIAL_ID = 'trial_01'   # auto-stamped by the generator; do not edit
import os

EXPERIMENT_DIR = os.getcwd()
SPACE_PATH = os.path.join(EXPERIMENT_DIR, 'space.json')
TRIALS_PATH = os.path.join(EXPERIMENT_DIR, 'trials.json')

MAX_TRIALS = 20    # campaign cap: stop generating new trials after this many (None = unlimited)

from ag_hypopt import AGHyperopt

opt = AGHyperopt()
opt.fit(SPACE_PATH, TRIALS_PATH)
proposed_trials = opt.propose_trials(10)


phase = uniform  (1/5 trials)
ID |     ei | params
 1 |      - | {"gamma_anneal": 0.27100474595421226, "lr_gamma": 0.8273927560856282, "lr_mu": 29.837966140304356, "sigma_ref": 21.650667689362617}
 2 |      - | {"gamma_anneal": 0.3731007177781912, "lr_gamma": 0.7661124332036362, "lr_mu": 9.237738384801235, "sigma_ref": 16.993106713845833}
 3 |      - | {"gamma_anneal": 0.5153797626017405, "lr_gamma": 0.883592600819902, "lr_mu": 15.461412199807526, "sigma_ref": 7.093692272874462}
 4 |      - | {"gamma_anneal": 0.052073235184952826, "lr_gamma": 0.8755329019643157, "lr_mu": 25.323728168932202, "sigma_ref": 16.908304190741344}
 5 |      - | {"gamma_anneal": 0.46411577354123024, "lr_gamma": 0.6430523056599491, "lr_mu": 17.54554848937635, "sigma_ref": 9.343383003309365}
 6 |      - | {"gamma_anneal": 0.3195897119418725, "lr_gamma": 0.9150598542529591, "lr_mu": 29.76081963740514, "sigma_ref": 24.130898126872285}
 7 |      - | {"gamma_anneal": 0.0076597207072358064, "lr_gamma": 0.7814145470521

## 3. ✍️ WRITE - Analyze and choose

Using the summary from cell 1 (previous trials + registry) and the candidate table from
cell 2, choose the ONE proposal that makes the most sense for the next trial. No hypothesis
and no expectation are needed: base the choice on what the trials have shown and on the
candidates themselves.

There are no previous trials and the registry is empty, so there is no objective or EI to compare against; every candidate is a uniform warm-up draw. As a neutral criterion I take the candidate closest to the middle of each parameter range. Candidate 2 sits nearest the centre: sigma_ref 10.8 of [5, 25], lr_mu 16.4 of [5, 30], lr_gamma 0.79 of [0.2, 1.0], gamma_anneal 0.35 of [0, 0.75], giving the smallest total scaled distance to the midpoints. I choose candidate 2 (2 goes into the SET cell below).


In [2]:
# 4. ✍️ SET - Chosen candidate (edit INDEX below, then run this cell)
INDEX = 2            # [Put the index here: 1-based candidate number from the table in cell 2]


In [3]:
# 5. ▶ RUN - Execute the chosen trial  (do not interrupt unless obviously broken)
assert INDEX is not None, 'INDEX not set: edit the SET cell (cell 4) before running this one'
assert 1 <= INDEX <= len(proposed_trials), f'bad INDEX: {INDEX}'
CHOSEN = proposed_trials[INDEX - 1]['params']
print('chosen:', CHOSEN)

from ag_hypopt import objective

import traceback, time
t0 = time.time()
try:
    loss, uncertainty, report = objective(CHOSEN)
    print(f'trial finished in {(time.time()-t0)/60:.1f} min')
    print(f'objective = {loss:.4f} ± {uncertainty:.4f}')
    print(report)
except Exception:
    traceback.print_exc()
    loss, uncertainty = None, None


chosen: {'gamma_anneal': 0.348582398649244, 'lr_gamma': 0.7924933167748989, 'lr_mu': 16.41594101450036, 'sigma_ref': 10.825323440413898}
Running  1nW Trans05 ... μ 9.39 -> 8.40 | γ 8.5 -> 8.40 | NLL 1.10
Running  1nW Trans20 ... μ 17.32 -> 17.33 | γ 8.5 -> 8.43 | NLL 2.55
Running  1nW Trans60 ... μ 61.37 -> 61.43 | γ 8.5 -> 8.66 | NLL 2.85
Running 1nW Trans100 ... μ 70.82 -> 71.66 | γ 8.5 -> 8.55 | NLL 3.50
Running  3nW Trans05 ... μ 13.20 -> 11.68 | γ 14.1 -> 13.78 | NLL 1.56
Running  3nW Trans20 ... μ 34.28 -> 34.51 | γ 14.1 -> 14.32 | NLL 3.05
Running  3nW Trans60 ... μ 103.20 -> 103.89 | γ 14.1 -> 14.17 | NLL 3.35
Running 3nW Trans100 ... μ 175.71 -> 177.18 | γ 14.1 -> 14.23 | NLL 3.01

Total: 21.8 min
trial finished in 21.8 min
objective = 0.0016 ± 0.0010
exp           μ_true   μ_fit     Δμ% |   γ_true   γ_fit     Δγ%
1nW Trans05     9.39    8.40   -10.6 |     8.50    8.40    -1.2
1nW Trans20    17.32   17.33    +0.1 |     8.50    8.43    -0.8
1nW Trans60    61.37   61.43    +0.1 

## 6. ✍️ WRITE - Analyze the results

Record what actually happened in this trial: the measured objective, how the chosen
configuration behaved, and how that compares with the previous trials. No hypothesis to
confirm or refute, and no expectation to check: just the facts.

The first trial ran the 8-experiment benchmark in 21.8 minutes with the warm-up candidate (sigma_ref 10.8, lr_mu 16.4, lr_gamma 0.79, gamma_anneal 0.35; n_runs and n_iter fixed at 100). Measured objective 0.0016 +/- 0.0010. Both parameters are recovered well: mu RMSE 0.915 (relative 5.6 percent, bias +0.099) and gamma RMSE 0.164 (relative 1.4 percent, bias +0.019). Six of the eight experiments land essentially on the truth for both parameters (mu within about 1 percent, gamma within about 2 percent). The two exceptions are the low-count Trans05 experiments (1nW n_target 61, 3nW n_target 252), where mu comes out low by about 10.6 and 11.5 percent while gamma stays within about 2 percent. There are no previous trials and the registry was empty, so there is nothing to compare against yet.

**Summary:** trial_01 (first warm-up uniform draw: sigma_ref 10.8, lr_mu 16.4, lr_gamma 0.79, gamma_anneal 0.35; n_runs=n_iter=100) ran the 8-experiment benchmark in 21.8 min: objective 0.0016 +/- 0.0010, mu rel-RMSE 5.6 percent, gamma rel-RMSE 1.4 percent.
**Key insight:** The warm-up draw already recovers gamma to 1.4 percent and mu to 5.6 percent relative RMSE, with the only visible residual being the two low-count Trans05 experiments where mu lands about 10 to 11 percent low.

Structural-change proposals (score, likelihood, model, protocol) belong here as notes for
Anuar: describe them, never act on them, never stop the campaign because of them.
No structural-change notes for this trial.


In [4]:
# 7. ✍️ SET - Summary and key insight for the record (edit the strings below, then run)
SUMMARY = 'trial_01 (first warm-up uniform draw: sigma_ref 10.8, lr_mu 16.4, lr_gamma 0.79, gamma_anneal 0.35; n_runs=n_iter=100) ran the 8-experiment benchmark in 21.8 min: objective 0.0016 +/- 0.0010, mu rel-RMSE 5.6 percent, gamma rel-RMSE 1.4 percent.'      # [Put the one-line summary here: copy it from cell 6]
KEY_INSIGHT = 'The warm-up draw already recovers gamma to 1.4 percent and mu to 5.6 percent relative RMSE, with the only visible residual being the two low-count Trans05 experiments where mu lands about 10 to 11 percent low.'  # [Put the one-sentence key insight here: copy it from cell 6]


In [5]:
# 8. ▶ RUN - Record the trial in trials.json
import json

assert SUMMARY is not None, 'SUMMARY not set: fill the SET cell (cell 7) first'
assert KEY_INSIGHT is not None, 'KEY_INSIGHT not set: fill the SET cell (cell 7) first'

entry = {
    'trial_id': TRIAL_ID,
    'config': CHOSEN,
    'objective': loss,
    'uncertainty': uncertainty,
    'summary': SUMMARY,
    'key_insight': KEY_INSIGHT,
    'notebook': f'{TRIAL_ID}.ipynb',
}
data = json.load(open(TRIALS_PATH))
assert not any(t.get('trial_id') == TRIAL_ID for t in data['trials']),     f'{TRIAL_ID} is already registered in trials.json'
data['trials'].append(entry)
json.dump(data, open(TRIALS_PATH, 'w'), indent=2)
best = min((t for t in data['trials'] if t.get('objective') is not None),
           key=lambda t: t['objective'], default=None)
print('saved', TRIAL_ID, '| current best:', best['trial_id'] if best else None)


saved trial_01 | current best: trial_01


In [2]:
# 9. ▶ RUN - Generate the next trial (or end the campaign)
import os, re, json as _json

num = int(re.search(r'(\d+)$', TRIAL_ID).group(1))
nxt = num + 1
if MAX_TRIALS is not None and nxt > MAX_TRIALS:
    print(f'Campaign complete: cap MAX_TRIALS={MAX_TRIALS} reached after {TRIAL_ID}. Stop here.')
else:
    template_path = os.path.join(EXPERIMENT_DIR, 'template.ipynb')
    target = os.path.join(EXPERIMENT_DIR, f'trial_{nxt:02d}.ipynb')
    assert not os.path.exists(target), f'{target} already exists: refusing to overwrite'

    nb = _json.load(open(template_path))

    def _stamp(old, new):
        for c in nb['cells']:
            if old in ''.join(c.get('source', [])):
                c['source'] = [s.replace(old, new) for s in c['source']]
                return True
        raise RuntimeError(f'stamp target {old!r} not found in the template')

    _stamp('{N}', f'{nxt:02d}')
    _stamp('trial_XXX', f'trial_{nxt:02d}')
    _json.dump(nb, open(target, 'w'), indent=1)
    print(f'Created {os.path.basename(target)}. Open it and follow its cells from the top.')


Created trial_02.ipynb. Open it and follow its cells from the top.
